# 02 · Campaign — graft CDRs onto candidate human frameworks → variants CSV

**Standard slot:** *design campaign.* **For Project 16 this means:** run the humanization campaign —
graft the non-human CDRs onto **several candidate human germline frameworks**, generate variants with
and without **Vernier back-mutations** (and a resurfacing variant as the `[extension]` alternative),
score humanness + the ΔΔG proxy, and write a variants CSV (D2).

**Compute reality (be honest):** this project is **light** — humanness scoring, an IgFold/ImmuneBuilder
Fv model, and ΔΔG proxies are **free-tier Colab T4 friendly**. No A100 needed. This notebook runs on the
**mock** backend so the plumbing executes anywhere; switch each `tool="mock"` to the real backend
(AbLang / IgFold / FoldX-Rosetta) on Colab when ready.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (rule 4)

Tools change. Before a real run, confirm the pinned upstream repos still exist and **pin the exact
commit** you used (put it in `LOG.md`). This HTTP-checks the URLs; it does not install anything. Mark
the OASis/Hu-mAb/T20 humanness backends as **"verify current public release/host"** — they move.

In [ ]:
import requests

# Pinned upstreams for the humanization family (pin the COMMIT you actually use — these move).
UPSTREAMS = {
    "AbLang (antibody LM — humanness + restoration)": "https://github.com/oxpig/AbLang",
    "ImmuneBuilder / IgFold-style Fv (for ΔΔG model)": "https://github.com/oxpig/ImmuneBuilder",
    "ProteinMPNN (framework optimization [extension])": "https://github.com/dauparas/ProteinMPNN",
    "BioPhi (OASis / Hu-mAb humanness — VERIFY host)": "https://github.com/Merck/BioPhi",
}
for name, url in UPSTREAMS.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=10)
        print(f"[{r.status_code}] {name}\n        {url}")
    except Exception as e:
        print(f"[ERR] {name}: {e}\n        {url}")
print("\nNOTE: OASis/Hu-mAb (BioPhi) + T20 server — VERIFY the current public release/host at course")
print("start and pin it (MANUAL.md §2). FoldX/Rosetta for ΔΔG are licensed — see MANUAL.md §2.")
print("Pin the exact COMMIT/tag of each tool in LOG.md before any real campaign.")

## Campaign parameters — candidate human frameworks

The CDRs (binding) are **fixed**; the **framework** is the design variable. Real humanization tries
**several candidate human germline frameworks** (the closest human germlines to the parental, by V/J
gene) and keeps the one that best balances humanness and stability. Here we mock a small panel of
candidate frameworks; on a real run, pull the actual IMGT germlines you chose in notebook 01.

In [ ]:
from humanization_tools import NONHUMAN_AB, HUMAN_FRAMEWORK, parental_sequence

parent = parental_sequence(NONHUMAN_AB)

# Candidate human germline frameworks. The first is the package placeholder; the others are small,
# deterministic perturbations standing in for DIFFERENT human germlines (e.g., IGHV1 vs IGHV3 vs IGHV4
# family members). REPLACE with the real IMGT germline FRs you selected in notebook 01.
def _variant_framework(base, name, swaps):
    fw = dict(base); fw["name"] = name
    for region, (pos, aa) in swaps.items():
        s = list(fw[region])
        if 0 <= pos < len(s):
            s[pos] = aa
        fw[region] = "".join(s)
    return fw

CANDIDATE_FRAMEWORKS = [
    HUMAN_FRAMEWORK,  # human_IGHV_teaching_placeholder
    _variant_framework(HUMAN_FRAMEWORK, "human_IGHV1-like_teaching", {"FR1": (5, "Q"), "FR3": (10, "K")}),
    _variant_framework(HUMAN_FRAMEWORK, "human_IGHV4-like_teaching", {"FR2": (3, "Q"), "FR3": (5, "N")}),
]
print("candidate human frameworks (VERIFY/replace with real IMGT germlines):")
for fw in CANDIDATE_FRAMEWORKS:
    print("  -", fw["name"])
print("\nTOOL = 'mock' (switch to 'ablang' on Colab T4 for real humanness-aware grafting)")

## Run the campaign (mock) → variants

For **each** candidate framework we generate three variants:
1. **bare CDR graft** (maximally human, expect high ΔΔG),
2. **graft + Vernier back-mutations** (rescue stability at a small humanness cost),
3. a **resurfacing** variant `[extension]` (surface FR residues only — fewer mutations, less human).

We also add the two mandatory **controls** the validation plan needs:
- the **parental** antibody (non-human; humanness floor, stability ceiling), and
- an **over-humanized decoy** (humanize aggressively *including* the Vernier zone — high humanness,
  expected to LOSE binding/stability; the negative control for "over-humanization").

In [ ]:
import pandas as pd
from humanization_tools import (graft_cdrs, resurface, vernier_backmutations, score_variants,
                                  HumanizedVariant)

variants = []
for fw in CANDIDATE_FRAMEWORKS:
    # 1) bare graft
    g = graft_cdrs(NONHUMAN_AB, fw, scheme="kabat", tool="mock",
                   variant_id=f"EXAMPLE_DATA_{fw['name']}_graft")
    # 2) graft + Vernier back-mutations
    bms = vernier_backmutations(g, NONHUMAN_AB, fw)
    tokens = tuple(b["token"] for b in bms)
    g_bm = graft_cdrs(NONHUMAN_AB, fw, back_mutations=tokens, tool="mock",
                      variant_id=f"EXAMPLE_DATA_{fw['name']}_graft_BM")
    variants.extend([g, g_bm])

# 3) resurfacing alternative [extension]
veneer = resurface(NONHUMAN_AB, tool="mock", variant_id="EXAMPLE_DATA_resurface")
variants.append(veneer)

# --- Controls ---
# parental (non-human) control: humanness floor + stability ceiling (ΔΔG = 0 by definition).
parental_ctrl = HumanizedVariant(
    variant_id="EXAMPLE_DATA_parental_control", sequence=parent, method="parental", tool="mock",
    cdr1=NONHUMAN_AB["CDR1"], cdr2=NONHUMAN_AB["CDR2"], cdr3=NONHUMAN_AB["CDR3"],
    synthetic=True, notes=["control: parental non-human antibody"])
# over-humanized decoy: graft AND humanize the Vernier zone too (no back-mutations) — expected to lose
# binding/stability. The negative control for over-humanization.
decoy = graft_cdrs(NONHUMAN_AB, HUMAN_FRAMEWORK, back_mutations=(), tool="mock",
                   variant_id="EXAMPLE_DATA_over_humanized_decoy")
decoy.method = "over_humanized_decoy"
decoy.notes.append("control: over-humanized decoy (no Vernier rescue) — expected to lose binding")
variants.extend([parental_ctrl, decoy])

score_variants(variants, parent, tool="mock")
print(f"generated {len(variants)} variants (grafts + back-mutated + resurface + 2 controls)")

In [ ]:
rows = [v.as_row() for v in variants]
camp = pd.DataFrame(rows)
cols = ["variant_id", "method", "tool", "parental", "human_framework", "scheme",
        "cdr1", "cdr2", "cdr3", "n_framework_mutations",
        "oasis_like", "t20_like", "germline_id", "ddg_kcal_mol", "synthetic"]
camp = camp[[c for c in cols if c in camp.columns]]
camp["cdr3_len"] = camp["cdr3"].str.len()
camp["n_back_mutations"] = [len(v.back_mutations) for v in variants]
camp.to_csv("results/campaign.csv", index=False)
print("wrote results/campaign.csv", camp.shape)
print("SYNTHETIC?", bool(camp["synthetic"].all()), "(mock => all numbers are EXAMPLE_DATA)")
camp[["variant_id", "method", "n_framework_mutations", "n_back_mutations",
      "oasis_like", "ddg_kcal_mol"]]

## Quick campaign sanity look — the trade-off is already visible

Before filtering, eyeball the two axes that matter: **humanness** vs the **ΔΔG proxy**, coloured by
method. The parental control sits at low humanness / zero ΔΔG; bare grafts at high humanness / high ΔΔG;
back-mutated grafts and resurfacing in between. On **mock** these are SYNTHETIC and only show the
plumbing; on a real run this is your headline figure.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].hist(camp["n_framework_mutations"], bins=10)
ax[0].set_title("framework mutations / variant"); ax[0].set_xlabel("# FR residues changed")
# humanness vs ΔΔG scatter, coloured by method
for method, sub in camp.groupby("method"):
    ax[1].scatter(sub["oasis_like"], sub["ddg_kcal_mol"], label=method, s=40)
ax[1].set_xlabel("humanness (OASis-like, heuristic)")
ax[1].set_ylabel("ΔΔG proxy (>0 destabilizing)")
ax[1].set_title("humanness vs ΔΔG (SYNTHETIC mock)")
ax[1].legend(fontsize=7)
plt.suptitle("Campaign pool — SYNTHETIC (mock) distributions; for plumbing only")
plt.tight_layout(); plt.savefig("results/campaign_distributions.png", dpi=150); plt.show()
print("Reminder: mock values are SYNTHETIC — real shape comes from OASis/Hu-mAb + FoldX/Rosetta.")

## D2 checklist
- [ ] `results/campaign.csv`: the variant pool (method, framework, CDRs, humanness + ΔΔG proxy, back-
      mutation count), one row per variant — including the **parental** and **over-humanized decoy**
      controls.
- [ ] Several **candidate human germline frameworks** tried (not just one).
- [ ] Version-verify cell run; exact tool **commits** pinned in `LOG.md`.
- [ ] Design log: parental antibody, candidate frameworks, scheme (Kabat/Chothia/IMGT), seed,
      tool/version, runtime.
- [ ] (Real run) humanness via OASis/Hu-mAb/T20/AbLang + ΔΔG via FoldX/Rosetta on an IgFold model;
      note that grafting typically LOSES affinity/stability and needs back-mutations.
- [ ] 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — the shared antibody filter (`design_type="antibody"`).